# Shrunk $\pi$: focused same-anchor follow-up

Linear and sigmoid are separate experimental conditions. This notebook reads immutable artifacts only. Interim values are descriptive; each schedule's nominal two-sided test appears only after its 352 paired replicas are complete. The two tests do not constitute a familywise-5% claim.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'mnist_experiment/plan11.md').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Run this notebook from within the repository')
sys.path.insert(0, str(ROOT))
from mnist_experiment.rotated_mnist.plan11.followup_analysis import load_progress

progress = load_progress(ROOT)


## Completion and integrity

Only fully validated fixed-versus-blend pairs enter these displays. Missing pairs remain visible as incomplete; a corrupt completed artifact raises an error.


In [ ]:
summary = []
for schedule, result in progress['schedules'].items():
    summary.append({
        'schedule': schedule,
        'validated pairs': result['completed_pairs'],
        'planned pairs': result['planned_pairs'],
        'status': result['status'],
        'descriptive mean gain': result['descriptive_mean_gain'],
        'descriptive median gain': result['descriptive_median_gain'],
        'positive pairs': result['positive_pairs'],
        'mean pair seconds': result['mean_pair_wall_seconds'],
        'max blend fallback': result['maximum_blend_fallback_fraction'],
    })
display(pd.DataFrame(summary).set_index('schedule').style.format({'descriptive mean gain': '{:+.4f}', 'descriptive median gain': '{:+.4f}', 'mean pair seconds': '{:.1f}', 'max blend fallback': '{:.1%}'}, na_rep='-'))

colors = {'linear': '#147d75', 'sigmoid': '#c3653e'}
fig, axes = plt.subplots(2, 1, figsize=(10, 6), layout='constrained')
for ax, (schedule, result) in zip(axes, progress['schedules'].items()):
    rows = result['rows']
    if rows:
        x = np.array([row['replica_index'] for row in rows])
        gain = np.array([row['gain'] for row in rows])
        ax.scatter(x, gain, s=18, alpha=.45, color=colors[schedule], edgecolors='none', label='Paired gain')
        ax.plot(x, np.cumsum(gain) / np.arange(1, len(gain) + 1), color=colors[schedule], lw=2, label='Descriptive running mean')
        ax.set_xlim(0, min(352, max(16, int(x.max()) + 8)))
    else:
        ax.set_xlim(0, 16)
    ax.axhline(0, color='#4c5560', lw=.9)
    ax.set_title(f'{schedule.capitalize()}  |  {result["completed_pairs"]}/352 validated pairs', loc='left')
    ax.set_ylabel('Fixed minus blend NLL-AUC')
    ax.grid(axis='y', color='#d8dcdf', lw=.6)
    ax.spines[['top', 'right']].set_visible(False)
axes[-1].set_xlabel('Predeclared replica index')
axes[0].legend(frameon=False, loc='upper right') if progress['schedules']['linear']['rows'] else None
plt.show()


## Mean predictive trajectories

Every curve averages the same validated fixed-versus-blend replicas within its schedule. NLL and environmental accuracy use the stored pre-update exposure clock; running AUC uses the terminal summary's trapezoidal convention. Partial-sample curves are descriptive, not tests.


In [ ]:
from matplotlib.ticker import PercentFormatter

policy_colors = {'fixed': '#42596b', 'blend': '#c3653e'}
for suffix, heading, column_titles in (
    ('', 'Mean current-environment trajectories', ('NLL', 'Accuracy')),
    ('_auc', 'Running exposure-normalized AUC', ('NLL AUC', 'Accuracy AUC')),
):
    fig, axes = plt.subplots(2, 2, figsize=(11, 6.5), layout='constrained')
    fig.suptitle(heading, fontsize=14)
    legend_ax = None
    for row_index, (schedule, result) in enumerate(progress['schedules'].items()):
        curve = result['mean_trajectory']
        for column_index, (field, label) in enumerate(zip(('nll', 'accuracy'), column_titles)):
            ax = axes[row_index, column_index]
            ax.set_title(f'{schedule.capitalize()} | {label}', loc='left', fontsize=10)
            ax.set_xlabel('Observations before evaluation') if row_index == 1 else None
            ax.grid(axis='y', color='#dfe4e7', linewidth=.7)
            ax.spines[['top', 'right']].set_visible(False)
            if curve is None:
                ax.text(.5, .5, 'Awaiting a complete pair', transform=ax.transAxes, ha='center', va='center', color='#65717a')
                continue
            x = np.asarray(curve['exposure'])
            for policy, name in (('fixed', 'Fixed .025'), ('blend', 'Blend (.025, .025)')):
                ax.plot(x, curve[f'{policy}_{field}{suffix}'], color=policy_colors[policy], linewidth=1.9, label=name)
            for boundary in (len(x) // 3, 2 * len(x) // 3):
                ax.axvline(x[boundary], color='#b7bec4', linewidth=.8, linestyle=':')
            ax.set_xlim(x[0], x[-1])
            if field == 'accuracy':
                ax.yaxis.set_major_formatter(PercentFormatter(1))
            legend_ax = legend_ax or ax
    if legend_ax is not None:
        legend_ax.legend(frameon=False, loc='best')
    plt.show()


## Fixed-size schedule-specific inference

A schedule appears here only when all 352 predeclared pairs have passed integrity checks. No interim $p$-values or confidence intervals are computed.


In [ ]:
finished = []
for schedule, result in progress['schedules'].items():
    if result['inference'] is None:
        display(Markdown(f'**{schedule.capitalize()}: incomplete** ({result["completed_pairs"]}/352 pairs). No inferential result yet.'))
    else:
        finished.append({'schedule': schedule, **result['inference']})
if finished:
    display(pd.DataFrame(finished).set_index('schedule').style.format({'mean_gain': '{:+.4f}', 'standard_deviation': '{:.4f}', 'confidence_low': '{:+.4f}', 'confidence_high': '{:+.4f}', 'two_sided_p': '{:.4g}'}))
    display(Markdown('Intervals and $p$-values are **unadjusted, schedule-specific**. A positive result does not establish a 5%-error-rate claim that at least one schedule works.'))


## Phase 2 discovery context

The original grid and stopped decision are separate from the fresh follow-up. They do not enter the intervals above.


In [ ]:
import json
decisions = ROOT / 'cache/mnist_experiment/rotated_mnist/plan11/decisions'
selection = json.loads((decisions / 'linear_selection.json').read_text())
development = json.loads((decisions / 'development_decision.json').read_text())
if selection['study_hash'] != development['study_hash'] or development['status'] != 'stopped':
    raise ValueError('Incompatible or incomplete Phase 2 decision artifacts')
display(Markdown(f'Original Plan 11 status: **{development["status"]}** ({development["reason"]}). The same-anchor follow-up changes the comparator, not this historical decision.'))
grid = pd.DataFrame(selection['grid'])
display(grid.pivot(index='anchor', columns='gain', values='nll_auc').style.format('{:.3f}').set_caption('Exploratory linear NLL-AUC by anchor and blend gain'))
